# **ICX360: explaining an LLM answer through parts of its context**

### Tutorial for the lesson "Neuropedia"

Neuronpedia shows what happens **inside** a model: which SAE features activate on a token, which tokens a feature responds to most strongly, which logits it raises. Often a different question matters: which part of the **input** did the model rely on when it produced a particular answer? If the context has six sentences and the model answered from one of them, which one was it? Explaining an answer through parts of the context is called in-context explainability, and ICX360 is an open IBM Research library of such methods.

What ICX360 is (checked on 11.10.2026):

- repository [github.com/IBM/ICX360](https://github.com/IBM/ICX360), Apache 2.0 license, [documentation](https://ibm.github.io/ICX360/), paper about the library: Wei et al., [ICX360: In-Context eXplainability 360 Toolkit](https://arxiv.org/abs/2511.10879), 2025;
- the library has three methods:
  1. **MExGen** ([Multi-Level Explanations for Generative Language Models](https://arxiv.org/abs/2403.14459)) removes parts of the context, measures how much the probability of the original answer drops, and fits a linear surrogate model to these drops. Its variants C-LIME and L-SHAP are versions of LIME and SHAP for a generative model;
  2. **CELL** ([CELL your Model: Contrastive Explanations for Large Language Models](https://arxiv.org/abs/2406.11785)) searches for an edit of the prompt after which the answer changes according to a chosen measure. The edits come from a second infiller model (T5 or BART), so the method is heavier and this notebook leaves it out;
  3. **Token Highlighter** ([Token Highlighter: Inspecting and Mitigating Jailbreak Prompts for Large Language Models](https://arxiv.org/abs/2412.18171)) is a gradient method: the importance of a prompt fragment is the norm of the gradient of the loss of a target answer with respect to the embeddings of the fragment's tokens. The authors proposed it for finding jailbreak fragments, but the target answer can be anything.

The plan: the open model Qwen2.5-0.5B-Instruct answers a question about a short text of six sentences. We explain its answer in three ways — C-LIME, L-SHAP and Token Highlighter — and check the explanation by intervention: we remove the most important sentence and see what the model answers now.

The notebook runs on CPU; no GPU is needed.

## Installation

ICX360 declares Python below 3.13, `openai<2` and `datasets<4` in its requirements, while Colab now has Python 3.13, `openai` 2 and `datasets` 4. The methods in this notebook need neither `openai` nor `datasets`, so we install the package without dependencies (`--no-deps --ignore-requires-python`) and pinned to a repository commit, and the two missing dependencies separately. The rest (`torch`, `spacy` with the `en_core_web_sm` model, `scikit-learn`, `sentence-transformers`) is already in Colab; the line with `transformers` and the spaCy model is for running outside Colab.

In [ ]:
%pip install -q "transformers==5.18.0" "sentence-transformers==5.7.0" "en_core_web_sm @ https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl"
%pip install -q evaluate==0.4.6 toma==1.1.0
%pip install -q --no-deps --ignore-requires-python "icx360 @ git+https://github.com/IBM/ICX360.git@0c971ce8ef82b3228a3a34fcd0f184044ab943a0"

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from icx360.algorithms.mexgen import CLIME, LSHAP
from icx360.algorithms.token_highlighter.th_llm import TokenHighlighter
from icx360.utils.model_wrappers import HFModel
from icx360.utils.general_utils import fix_seed, select_device

pd.set_option("display.max_colwidth", 100)
fix_seed(0)
device = select_device()
print(device)

## The model and the question

Qwen2.5-0.5B-Instruct is an instruct model with 0.5 billion parameters under the Apache 2.0 license. The weights are pinned to a Hugging Face commit so that the notebook's answers do not change when the model is updated.

One line below is required: `tokenizer.padding_side = "left"`. MExGen computes the probability of the answer for a batch of perturbed contexts of different lengths at once, pads the shorter ones and concatenates each context with the answer tokens without passing an attention mask to the model (this is how `ProbScalarizedModel` works in the pinned ICX360 version). Qwen pads on the right by default, and then the padding tokens land between the context and the answer: in our run with right padding, removing any single sentence dropped the mean log probability of the answer by 0.9–1.5, and the scores of all six sentences came out between 0.52 and 1.37. With left padding the extra tokens sit at the beginning and barely interfere.

The context is six sentences about the Eiffel Tower. One of them, the fifth, answers the question, and it can only be understood together with the fourth: "this title" in the fifth sentence is the title of the tallest structure from the fourth. The other sentences are unrelated to the question.

In [ ]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name, revision="7ae557604adf67be50417f59c2c2f167def9a775")
tokenizer.padding_side = "left"   # see the paragraph above: without this line MExGen scores are distorted
model = AutoModelForCausalLM.from_pretrained(model_name, revision="7ae557604adf67be50417f59c2c2f167def9a775", dtype=torch.float32).to(device).eval()

sentences = [
    "The Eiffel Tower was built for the 1889 World's Fair in Paris.",
    "Its designer, Gustave Eiffel, also worked on the internal frame of the Statue of Liberty.",
    "The tower is 330 metres tall.",
    "For 41 years it was the tallest man-made structure in the world.",
    "In 1930 the Chrysler Building in New York took this title.",
    "Today the tower receives about seven million visitors a year.",
]
context = " ".join(sentences)
question = "Which building took the title of the tallest man-made structure from the Eiffel Tower?"

`HFModel` is the ICX360 wrapper around a Hugging Face model: the explainers use it to generate text and to compute the probability of a given answer. We pass the prompt as a list of parts: then the template parts (`"Context: "`, the question) can be marked as fixed, and the explanation covers only the context. Generation is greedy (`do_sample=False`), so a rerun gives the same answer.

In [ ]:
wrapped_model = HFModel(model, tokenizer)
model_params = {
    "max_new_tokens": 32,
    "do_sample": False,
    "chat_template": True,
    "system_prompt": "Answer the question in one short sentence using only the provided context.",
}

def make_prompt(ctx):
    return ["Context: ", ctx, "\n\nQuestion: ", question, "\n\nAnswer: "]

prompt = make_prompt(context)
output_orig = wrapped_model.generate("".join(prompt), text_only=False, **model_params)
print(output_orig.output_text[0])

**Task 1.** Which building does the model name in its answer?

## MExGen C-LIME: which sentences matter for the answer

C-LIME works as follows:

1. it splits the context into sentences with the spaCy model `en_core_web_sm` (`segment_type="s"`);
2. it builds perturbed versions of the context: it removes each sentence on its own and each pair of sentences (`max_units_replace=2` by default). For six sentences this is $1 + 6 + 15 = 22$ versions including the original, and all of them are enumerated rather than sampled;
3. for each version it computes the `"prob"` scalarizer, the per-token mean log probability of the **original** answer given this context. The model does not generate anything new here;
4. it fits a linear model: the features are which sentences were removed, the target is the mean log probability. A sentence's score is how much this probability drops when the sentence is removed. A positive score means the original answer becomes less likely without the sentence.

In `unit_types`, the label `"p"` (paragraph) marks the context, which is split into sentences, and the label `"n"` (not of interest) marks the template parts: they are never removed and get no score.

In [ ]:
unit_types = ["n", "p", "n", "n", "n"]
ind_segment = [False, True, False, False, False]

fix_seed(0)
clime = CLIME(wrapped_model, scalarizer="prob", segmenter="en_core_web_sm")
out_clime = clime.explain_instance(prompt, unit_types, output_orig=output_orig, ind_segment=ind_segment,
                                   segment_type="s", model_params=model_params)

att = out_clime["attributions"]
clime_df = pd.DataFrame({"unit": att["units"], "type": att["unit_types"], "C-LIME": att["prob"]})
clime_df = clime_df[clime_df["type"] == "s"].reset_index(drop=True)
clime_df.round(3)

**Task 2.** Which sentence has the highest C-LIME score? How many times larger is its score than the score of the next sentence? Round to two decimals.

In [ ]:
# sort clime_df by the C-LIME column and compare the two top scores
...

## MExGen L-SHAP

L-SHAP (local SHAP) estimates Shapley values of the sentences: a sentence's contribution is averaged over different sets of other removed sentences instead of coming from a single linear model. For each sentence L-SHAP considers only its nearest neighbors (`num_neighbors`), so it is faster than exact SHAP. If C-LIME and L-SHAP agree, the explanation does not depend on how the drops are aggregated.

In [ ]:
fix_seed(0)
lshap = LSHAP(wrapped_model, scalarizer="prob", segmenter="en_core_web_sm")
out_lshap = lshap.explain_instance(prompt, unit_types, output_orig=output_orig, ind_segment=ind_segment,
                                   segment_type="s", model_params=model_params)

att = out_lshap["attributions"]
lshap_scores = [p for p, t in zip(att["prob"], att["unit_types"]) if t == "s"]
clime_df["L-SHAP"] = lshap_scores
clime_df.round(3)

**Task 3.** Is the sentence with the highest score the same for C-LIME and L-SHAP? What is its L-SHAP score (three decimals)?

In [ ]:
# find idxmax for each column
...

## Checking by intervention

A score is a claim about the model: "without the fifth sentence the original answer becomes less likely, more so than without any other sentence". It can be checked directly. We remove the fifth sentence, compute the mean log probability of the original answer with the same scalarizer, and look at what the model generates now. For comparison we do the same with the fourth sentence, without which "this title" in the fifth loses its referent.

In [ ]:
def without(i):
    return " ".join(s for j, s in enumerate(sentences) if j != i)

variants = {"full context": context, "without sentence 4": without(3), "without sentence 5": without(4)}
for name, ctx in variants.items():
    logp = clime.scalarized_model.scalarize_output(inputs=["".join(make_prompt(ctx))], ref_output=output_orig,
                                                    **model_params)
    answer = wrapped_model.generate("".join(make_prompt(ctx)), **model_params)[0]
    print(f"{name}: mean log probability of the original answer {logp.item():.3f}")
    print("   answer:", answer)

**Task 4.** Which structure does the model name when the fifth sentence is removed from the context? Which sentence of the context contains this name?

## Multi-level explanation: words inside the main sentence

The M in MExGen stands for multi-level. The sentence with the highest score can be split into words (`segment_type="w"`) while the others stay sentences. Then the explainer tells which words inside the main sentence carry the answer without spending model calls on words of unimportant sentences.

In [ ]:
units = out_clime["attributions"]["units"]
types = out_clime["attributions"]["unit_types"]
scores = np.array(out_clime["attributions"]["prob"])
ind_words = np.zeros(len(units), dtype=bool)
ind_words[np.argmax(scores)] = True

fix_seed(0)
out_words = clime.explain_instance(units, types, output_orig=output_orig, ind_segment=ind_words,
                                   segment_type="w", model_params=model_params)
att = out_words["attributions"]
words_df = pd.DataFrame({"unit": att["units"], "type": att["unit_types"], "C-LIME": att["prob"]})
words_df = words_df[words_df["type"] == "w"].reset_index(drop=True)
words_df.round(3)

**Task 5.** Which word of the fifth sentence got the highest C-LIME score, and what is that score (three decimals)? What is the score of the word "Building"?

In [ ]:
# sort words_df by score and find the row with the word Building
...

## Token Highlighter: a gradient instead of perturbations

MExGen calls the model dozens of times and looks only at the output; it is a black-box method. Token Highlighter does one forward and one backward pass: it takes a target answer (the `affirmation` argument), computes the cross-entropy of this answer given the prompt and takes the gradient with respect to the embeddings of the prompt tokens. With `pooling="mean_norm"` the score of a fragment is the norm of the mean gradient over the fragment's tokens.

We set the target answer to the model's original answer. Two differences from MExGen: Token Highlighter builds the prompt itself, through the chat template without a system prompt, and splits the whole user text into sentences, so "Context:" sticks to the first sentence and the question gets its own score.

In [ ]:
th = TokenHighlighter(model, tokenizer, segmenter="en_core_web_sm",
                      affirmation=output_orig.output_text[0], pooling="mean_norm")
user_text = "Context: " + context + "\n\nQuestion: " + question
out_th = th.explain_instance([user_text], ["p"], [True], "s")
th_df = pd.DataFrame({"unit": out_th["attributions"]["units"], "TH": out_th["attributions"]["scores"]})
th_df.round(4)

**Task 6.** Which fragment got the highest Token Highlighter score? How many times larger is its score than the second highest (two decimals)?

In [ ]:
# sort th_df by score
...

## ICX360 next to Neuronpedia

- ICX360 answers the question "which part of the input did this answer come from": the explanation is built at the level of text — sentences, phrases, words — and works the same way with any model that gives access to generation and probabilities (MExGen) or gradients (Token Highlighter).
- Neuronpedia answers the question "what does the model compute inside": SAE features, their activations on tokens and their contributions to logits. This needs the model weights and a trained SAE for the right layer.
- The two questions form a chain: ICX360 finds the input fragment the answer depends on (here the word "Chrysler" in the fifth sentence), and SAE features on the tokens of this fragment show what the model extracts from it. Checking by intervention is needed at both levels: remove the fragment from the input, as in task 4, or suppress a feature, as in steering on Neuronpedia.
- ICX360 scores are properties of one input and one answer. Task 4 showed that even methods that agree with each other do not say what the model will put in place of a removed fragment: without the fifth sentence it answers "the Statue of Liberty", although the score of the second sentence, where this name comes from, is small, and this becomes visible only after the intervention.